In [1]:
## Import Libraries
import pandas as pd

In [2]:
# Load the cleaned SEC Gig datasets
gig_edgar_sec = pd.read_csv('../../data/edgar_sec/gig_edgar_sec_all.csv')
gig_edgar_sec_10K_20F_only = pd.read_csv('../../data/edgar_sec/gig_edgar_sec_10K_20F_only.csv')

# Load the cleaned SP500 datasets
sp500_edgar_sec = pd.read_csv('../../data/edgar_sec/sp500_edgar_sec_all.csv')
sp500_edgar_sec_10K_20F_only = pd.read_csv('../../data/edgar_sec/sp500_edgar_sec_10K_20F_only.csv')

# Load the cleaned Yahoo Finance Gig datasets
gig_yfinance_quarterly = pd.read_csv('../../data/yfinance/gig_yfinance_quarterly.csv')
gig_yfinance_annual = pd.read_csv('../../data/yfinance/gig_yfinance_annually.csv')

# Load the cleaned Yahoo Finance SP500 datasets
sp500_yfinance_quarterly = pd.read_csv('../../data/yfinance/sp500_yfinance_quarterly.csv')
sp500_yfinance_annual = pd.read_csv('../../data/yfinance/sp500_yfinance_annually.csv')    

In [3]:
gig_edgar_sec.head()

,ticker,date,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,GrossProfit,IncomeTaxExpenseBenefit,...,WeightedAverageNumberOfSharesOutstandingBasic,company,Revenue,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares
0,AVON,2009-06-30,2009-07-30,2009,10-Q,Q2,5.237800e+09,5.237800e+09,5.237800e+09,5.237800e+09,...,5.237800e+09,AVON,5.237800e+09,1,1,0,1,1,1,1
1,AVON,2009-09-30,2009-10-29,2009,10-Q,Q3,7.882500e+09,7.882500e+09,7.882500e+09,7.882500e+09,...,7.882500e+09,AVON,7.882500e+09,1,1,0,1,1,1,1
2,AVON,2009-12-31,2010-02-25,2009,10-K,FY,9.938700e+09,9.938700e+09,9.938700e+09,9.938700e+09,...,9.938700e+09,AVON,9.938700e+09,1,1,0,1,1,1,1
3,AVON,2010-03-31,2010-04-30,2010,10-Q,Q1,2.186900e+09,2.186900e+09,2.186900e+09,2.186900e+09,...,2.186900e+09,AVON,2.186900e+09,1,1,0,1,1,1,1
4,AVON,2010-06-30,2010-07-29,2010,10-Q,Q2,4.665200e+09,4.665200e+09,4.665200e+09,4.665200e+09,...,4.665200e+09,AVON,4.665200e+09,1,1,0,1,1,1,1


In [4]:
# Combine company financials with stock price data for correlation analysis
def combine_financials_with_stock_prices_quarterly(financials_df, stock_prices_df):
    # Map fiscal period strings to integer quarters
    fiscal_to_quarter = {
        'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4,
        'FY': 4  # Treat annual filings as Q4
    }
    
    # yfinance colums names to lowercase to match financials_df
    stock_prices_df.columns = stock_prices_df.columns.str.lower()
    
    df_fin = financials_df.copy()
    df_fin['quarter'] = df_fin['fiscal_period'].map(fiscal_to_quarter)
    df_fin['year'] = df_fin['fiscal_year'].astype(int)

    # Normalize ticker casing to match yfinance
    df_fin['ticker'] = df_fin['ticker'].str.upper()

    combined_df = pd.merge(
        df_fin,
        stock_prices_df,
        on=['ticker', 'year', 'quarter'],
        how='left'
    )
    
    # remove cols that end with '_y' (from yfinance) that are duplicates of financials_df columns
    combined_df = combined_df[[col for col in combined_df.columns if not col.endswith('_y')]]
    
    # correct any remaining '_x' suffixes from financials_df
    combined_df.columns = [col.replace('_x', '') for col in combined_df.columns]
    
    # numeric cols last to avoid merge issues
    numeric_cols = [col for col in combined_df.columns if combined_df[col].dtype in ['float64', 'int64']]
    # lag_cols = [col for col in combined_df.columns if col.endswith('_lag1')]
    # numeric_cols += lag_cols
    other_cols = [col for col in combined_df.columns if col not in numeric_cols]
    combined_df = combined_df[other_cols + numeric_cols]

    return combined_df


In [5]:
# Combine datasets
gig_combined_quarterly = combine_financials_with_stock_prices_quarterly(gig_edgar_sec, gig_yfinance_quarterly)

# save combined dataset
gig_combined_quarterly.to_csv('../../data/merged_sec_yfin/gig_sec_yfin_quarterly.csv', index=False)

print(gig_combined_quarterly.columns)
gig_combined_quarterly

Index(['ticker', 'date', 'filed_date', 'form', 'fiscal_period', 'company',
       'industrykey', 'sectorkey', 'fiscal_year', 'Assets',
       'CommonStockSharesOutstanding', 'GrossProfit',
       'IncomeTaxExpenseBenefit', 'Liabilities', 'NetIncomeLoss',
       'OperatingExpenses', 'OperatingIncomeLoss',
       'WeightedAverageNumberOfSharesOutstandingBasic', 'Revenue',
       'has_gross_profit', 'has_operating_income', 'has_operating_expenses',
       'has_liabilities', 'has_revenue', 'has_common_stock_outstanding',
       'has_weighted_avg_shares', 'quarter', 'year', 'close', 'volume'],
      dtype='str')


,ticker,date,filed_date,form,fiscal_period,company,industrykey,sectorkey,fiscal_year,Assets,...,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares,quarter,year,close,volume
0,AVON,2009-06-30,2009-07-30,10-Q,Q2,AVON,NaN,NaN,2009,5.237800e+09,...,1,0,1,1,1,1,2,2009,NaN,NaN
1,AVON,2009-09-30,2009-10-29,10-Q,Q3,AVON,NaN,NaN,2009,7.882500e+09,...,1,0,1,1,1,1,3,2009,NaN,NaN
2,AVON,2009-12-31,2010-02-25,10-K,FY,AVON,NaN,NaN,2009,9.938700e+09,...,1,0,1,1,1,1,4,2009,NaN,NaN
3,AVON,2010-03-31,2010-04-30,10-Q,Q1,AVON,NaN,NaN,2010,2.186900e+09,...,1,0,1,1,1,1,1,2010,NaN,NaN
4,AVON,2010-06-30,2010-07-29,10-Q,Q2,AVON,NaN,NaN,2010,4.665200e+09,...,1,0,1,1,1,1,2,2010,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,UPWK,2024-12-31,2025-02-13,10-K,FY,UPWK,internet-content-information,communication-services,2024,4.579160e+08,...,1,1,1,1,1,1,4,2024,16.350000,1107900.0
450,UPWK,2025-03-31,2025-05-05,10-Q,Q1,UPWK,internet-content-information,communication-services,2025,1.467440e+08,...,1,1,1,1,1,1,1,2025,13.050000,1488900.0
451,UPWK,2025-06-30,2025-08-06,10-Q,Q2,UPWK,internet-content-information,communication-services,2025,2.960210e+08,...,1,1,1,1,1,1,2,2025,13.440000,2960800.0
452,UPWK,2025-09-30,2025-11-04,10-Q,Q3,UPWK,internet-content-information,communication-services,2025,4.463890e+08,...,1,1,1,1,1,1,3,2025,18.570000,2260100.0


In [6]:
sp500_combined_quarterly = combine_financials_with_stock_prices_quarterly(sp500_edgar_sec, sp500_yfinance_quarterly)
sp500_combined_quarterly.to_csv('../../data/merged_sec_yfin/sp500_sec_yfin_quarterly.csv', index=False)
print(sp500_combined_quarterly.columns)
sp500_combined_quarterly

Index(['ticker', 'date', 'filed_date', 'form', 'fiscal_period', 'company',
       'industrykey', 'sectorkey', 'fiscal_year', 'Assets',
       'CommonStockSharesOutstanding', 'IncomeTaxExpenseBenefit',
       'Liabilities', 'NetIncomeLoss', 'OperatingIncomeLoss',
       'WeightedAverageNumberOfSharesOutstandingBasic', 'GrossProfit',
       'OperatingExpenses', 'Revenue', 'has_gross_profit',
       'has_operating_income', 'has_operating_expenses', 'has_liabilities',
       'has_revenue', 'has_common_stock_outstanding',
       'has_weighted_avg_shares', 'quarter', 'year', 'close', 'volume'],
      dtype='str')


,ticker,date,filed_date,form,fiscal_period,company,industrykey,sectorkey,fiscal_year,Assets,...,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares,quarter,year,close,volume
0,A,2009-12-31,2009-12-21,10-K,FY,A,diagnostics-research,healthcare,2009,5.840000e+08,...,1,0,1,1,1,1,4.0,2009,19.710278,5224606.0
1,A,2010-03-31,2010-03-10,10-Q,Q1,A,diagnostics-research,healthcare,2010,2.400000e+07,...,1,0,1,1,1,1,1.0,2010,21.816435,5293387.0
2,A,2010-06-30,2010-06-07,10-Q,Q2,A,diagnostics-research,healthcare,2010,-2.300000e+07,...,1,0,1,1,1,1,2.0,2010,18.035511,5267804.0
3,A,2010-12-31,2010-12-20,10-K,FY,A,diagnostics-research,healthcare,2010,7.950000e+08,...,1,0,1,1,1,1,4.0,2010,26.282492,2050726.0
4,A,2011-03-31,2011-03-09,10-Q,Q1,A,diagnostics-research,healthcare,2011,9.400000e+07,...,1,0,1,1,1,1,1.0,2011,28.407673,3068470.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29644,ZTS,2024-12-31,2025-02-13,10-K,FY,ZTS,drug-manufacturers-specialty-generic,healthcare,2024,8.080000e+09,...,0,0,1,1,1,1,4.0,2024,160.109543,1327400.0
29645,ZTS,2025-03-31,2025-05-06,10-Q,Q1,ZTS,drug-manufacturers-specialty-generic,healthcare,2025,2.190000e+09,...,0,0,1,1,1,1,1.0,2025,162.288910,3137000.0
29646,ZTS,2025-06-30,2025-08-05,10-Q,Q2,ZTS,drug-manufacturers-specialty-generic,healthcare,2025,4.551000e+09,...,0,0,1,1,1,1,2.0,2025,154.232361,2826600.0
29647,ZTS,2025-09-30,2025-11-04,10-Q,Q3,ZTS,drug-manufacturers-specialty-generic,healthcare,2025,6.939000e+09,...,0,0,1,1,1,1,3.0,2025,145.192322,3736800.0


In [7]:
# Combine company financials with stock price data for correlation analysis
def combine_financials_with_stock_prices_annually(financials_df, stock_prices_df):
    # yfinance colums names to lowercase to match financials_df
    stock_prices_df.columns = stock_prices_df.columns.str.lower()
    
    df_fin = financials_df.copy()
    df_fin['year'] = df_fin['fiscal_year'].astype(int)

    # Normalize ticker casing to match yfinance
    df_fin['ticker'] = df_fin['ticker'].str.upper()

    combined_df = pd.merge(
        df_fin,
        stock_prices_df,
        on=['ticker', 'year'],
        how='left'
    )
    
    # remove cols that end with '_y' (from yfinance) that are duplicates of financials_df columns
    combined_df = combined_df[[col for col in combined_df.columns if not col.endswith('_y')]]
    
    # correct any remaining '_x' suffixes from financials_df
    combined_df.columns = [col.replace('_x', '') for col in combined_df.columns]
    
    # numeric cols last to avoid merge issues
    numeric_cols = [col for col in combined_df.columns if combined_df[col].dtype in ['float64', 'int64']]
    lag_cols = [col for col in combined_df.columns if col.endswith('_lag1')]
    numeric_cols += lag_cols
    other_cols = [col for col in combined_df.columns if col not in numeric_cols]
    combined_df = combined_df[other_cols + numeric_cols]

    return combined_df

In [8]:
# Combine annual datasets
gig_combined_annual = combine_financials_with_stock_prices_annually(gig_edgar_sec_10K_20F_only, gig_yfinance_annual)
gig_combined_annual

# save combined dataset
gig_combined_annual.to_csv('../../data/merged_sec_yfin/gig_sec_yfin_annual.csv', index=False)

print(gig_combined_annual.columns)
gig_combined_annual

Index(['ticker', 'date', 'filed_date', 'form', 'fiscal_period', 'company',
       'industrykey', 'sectorkey', 'fiscal_year', 'Assets',
       'CommonStockSharesOutstanding', 'GrossProfit',
       'IncomeTaxExpenseBenefit', 'Liabilities', 'NetIncomeLoss',
       'OperatingExpenses', 'OperatingIncomeLoss',
       'WeightedAverageNumberOfSharesOutstandingBasic', 'Revenue',
       'has_gross_profit', 'has_operating_income', 'has_operating_expenses',
       'has_liabilities', 'has_revenue', 'has_common_stock_outstanding',
       'has_weighted_avg_shares', 'year', 'close', 'volume'],
      dtype='str')


,ticker,date,filed_date,form,fiscal_period,company,industrykey,sectorkey,fiscal_year,Assets,...,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares,year,close,volume
0,AVON,2009-12-31,2010-02-25,10-K,FY,AVON,NaN,NaN,2009,9.938700e+09,...,1,1,0,1,1,1,1,2009,NaN,NaN
1,AVON,2010-12-31,2011-02-24,10-K,FY,AVON,NaN,NaN,2010,1.050750e+10,...,1,1,0,1,1,1,1,2010,NaN,NaN
2,AVON,2011-12-31,2012-02-29,10-K,FY,AVON,NaN,NaN,2011,1.020520e+10,...,1,1,0,1,1,1,1,2011,NaN,NaN
3,AVON,2012-12-31,2013-02-28,10-K,FY,AVON,NaN,NaN,2012,1.086280e+10,...,1,1,0,1,1,1,1,2012,NaN,NaN
4,AVON,2013-12-31,2014-02-26,10-K,FY,AVON,NaN,NaN,2013,1.109950e+10,...,1,1,0,1,1,1,1,2013,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,UPWK,2021-12-31,2022-02-15,10-K,FY,UPWK,internet-content-information,communication-services,2021,2.124180e+08,...,1,1,1,1,1,1,1,2021,34.160000,672300.0
118,UPWK,2022-12-31,2023-02-16,10-K,FY,UPWK,internet-content-information,communication-services,2022,2.693610e+08,...,1,1,1,1,1,1,1,2022,10.440000,1120800.0
119,UPWK,2023-12-31,2024-02-15,10-K,FY,UPWK,internet-content-information,communication-services,2023,3.672890e+08,...,1,1,1,1,1,1,1,2023,14.870000,1535700.0
120,UPWK,2024-12-31,2025-02-13,10-K,FY,UPWK,internet-content-information,communication-services,2024,4.579160e+08,...,1,1,1,1,1,1,1,2024,16.350000,1107900.0


In [9]:
# Combine annual datasets
sp500_combined_annual = combine_financials_with_stock_prices_annually(sp500_edgar_sec_10K_20F_only, sp500_yfinance_annual)
sp500_combined_annual

# save combined dataset
sp500_combined_annual.to_csv('../../data/merged_sec_yfin/sp500_sec_yfin_annual.csv', index=False)

print(sp500_combined_annual.columns)
sp500_combined_annual

Index(['ticker', 'date', 'filed_date', 'form', 'fiscal_period', 'company',
       'industrykey', 'sectorkey', 'fiscal_year', 'Assets',
       'CommonStockSharesOutstanding', 'IncomeTaxExpenseBenefit',
       'Liabilities', 'NetIncomeLoss', 'OperatingIncomeLoss',
       'WeightedAverageNumberOfSharesOutstandingBasic', 'GrossProfit',
       'OperatingExpenses', 'Revenue', 'has_gross_profit',
       'has_operating_income', 'has_operating_expenses', 'has_liabilities',
       'has_revenue', 'has_common_stock_outstanding',
       'has_weighted_avg_shares', 'year', 'close', 'volume'],
      dtype='str')


,ticker,date,filed_date,form,fiscal_period,company,industrykey,sectorkey,fiscal_year,Assets,...,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares,year,close,volume
0,A,2009-12-31,2009-12-21,10-K,FY,A,diagnostics-research,healthcare,2009,5.840000e+08,...,0,1,0,1,1,1,1,2009,19.710278,5224606.0
1,A,2010-12-31,2010-12-20,10-K,FY,A,diagnostics-research,healthcare,2010,7.950000e+08,...,0,1,0,1,1,1,1,2010,26.282492,2050726.0
2,A,2011-12-31,2011-12-16,10-K,FY,A,diagnostics-research,healthcare,2011,4.481000e+09,...,0,1,0,1,1,1,1,2011,22.159000,1930918.0
3,A,2012-12-31,2012-12-20,10-K,FY,A,diagnostics-research,healthcare,2012,5.444000e+09,...,0,1,0,1,1,1,1,2012,26.231359,4707905.0
4,A,2013-12-31,2013-12-19,10-K,FY,A,diagnostics-research,healthcare,2013,1.071000e+09,...,0,1,0,1,1,1,1,2013,37.020714,1316077.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7503,ZTS,2021-12-31,2022-02-15,10-K,FY,ZTS,drug-manufacturers-specialty-generic,healthcare,2021,6.260000e+09,...,0,0,0,1,1,1,1,2021,233.593536,1073400.0
7504,ZTS,2022-12-31,2023-02-14,10-K,FY,ZTS,drug-manufacturers-specialty-generic,healthcare,2022,6.675000e+09,...,0,0,0,1,1,1,1,2022,141.311722,1249500.0
7505,ZTS,2023-12-31,2024-02-13,10-K,FY,ZTS,drug-manufacturers-specialty-generic,healthcare,2023,7.776000e+09,...,0,0,0,1,1,1,1,2023,192.051224,1007200.0
7506,ZTS,2024-12-31,2025-02-13,10-K,FY,ZTS,drug-manufacturers-specialty-generic,healthcare,2024,8.080000e+09,...,0,0,0,1,1,1,1,2024,160.109543,1327400.0
